In [1]:
import chromadb
from chromadb.utils import embedding_functions

# 임베딩 모델
model_name = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"

embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name = model_name
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [2]:
# chromadb 접속
client_db = chromadb.HttpClient(
    host="127.0.0.1", port=28000
)

# 컬렉션 생성
collection = client_db.get_or_create_collection(
    name="book_collection1", embedding_function=embedding_func
)

# 데이터 추가하기
# documents => 벡터 검색 (유사도)
# metadatas => 조건 검색
# ids => 기본키 (자동생성 없음, 자체적으로 고유하게 만들어야 함)
collection.add(
    documents = [
        "파이썬 머신러닝 완벽 가이드는 머신러닝의 기초를 설명하는 책입니다.",
        "react는 웹 화면을 만드는 것을 설명하는 책입니다."
    ],
    metadatas = [
        {
            "title":"파이썬 머신러닝 가이드",
            "img":"./images/book1.jpg",
            "category":"AI"
        },
        {
            "title":"리액트 가이드",
            "img":"./images/book2.jpg",
            "category":"WEB"
        }
    ],
    ids = ["b1", "b2"]
)

In [3]:
# 조회
collection = client_db.get_collection(
    name="book_collection1", embedding_function=embedding_func
)

result = collection.query(
    query_texts = ["react"],
    # where = {"category":"AI"},
    n_results=1
)

result

{'ids': [['b2']],
 'distances': [[0.583158]],
 'embeddings': None,
 'metadatas': [[{'title': '리액트 가이드',
    'img': './images/book2.jpg',
    'category': 'WEB'}]],
 'documents': [['react는 웹 화면을 만드는 것을 설명하는 책입니다.']],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [4]:
import pandas as pd

df = pd.read_csv("./csv/OpenData_PotOpenTabletIdntfcC20260609.csv")
df.head()

C:\Users\pknukdt\AppData\Local\Temp\ipykernel_17000\3853337997.py:3: DtypeWarning: Columns (13,14,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("./csv/OpenData_PotOpenTabletIdntfcC20260609.csv")


,품목일련번호,품목명,업소일련번호,업소명,성상,큰제품이미지,표시앞,표시뒤,의약품제형,색상앞,...,표기이미지앞,표기이미지뒤,표기코드앞,표기코드뒤,변경일자,사업자번호,품목영문명,보험코드,표준코드,Unnamed: 33
0,200808877,페라트라정2.5밀리그램(레트로졸),19560004,(주)유한양행,어두운황색의원형필름코팅정,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...,YH,LT,원형,노랑,...,-,-,-,-,-,1188100601,-,-,8806421038804|8806421038811|8806421038828,NaN
1,200808948,졸뎀속붕정(졸피뎀타르타르산염),20080422,보령제약(주),흰색의원형구강붕해정제,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...,-,-,원형,하양,...,-,-,-,-,-,1348502031,-,-,8806419054205|8806419054212|8806419054229,NaN
2,200809076,가스프렌정(모사프리드시트르산염이수화물),19760002,경동제약(주),분할선을가진흰색의장방형필름코팅정제,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...,KD,분할선,장방형,하양,...,-,-,-,-,20211223,1248135518,GasprenTab,648102540,8806481025400|8806481025417|8806481025424|8806...,NaN
3,200809361,바르탄정(발사르탄),19870002,(주)휴온스,엷은적색의원형필름코팅정,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...,V분할선T,HS8,원형,분홍,...,-,-,-,-,-,8638700270,-,-,8806706056509|8806706056516|8806706056523,NaN
4,200809402,리피논정80밀리그램(아토르바스타틴칼슘삼수화물),19540001,동아에스티(주),이약은흰색의타원형필름코팅정이다.,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...,LPT,80,타원형,하양,...,-,-,-,-,20260303,2048640122,Lipinontablet80mg(Atorvastatincalciumtrihydrate),642506260,8806425062607|8806425062614|8806425062621|8806...,NaN


In [5]:
df.columns[12:18]

Index(['분할선뒤', '크기장축', '크기단축', '크기두께', '이미지생성일자(약학정보원)', '분류번호'], dtype='object')

In [6]:
collection = client_db.get_or_create_collection(
    name="test1", embedding_function=embedding_func
)

batch_size = 5000

# 전체 데이터 27000개를 생성하기
documents = df['품목명'].tolist()
metadatas = [
    {
        "title": row['분류번호'],
        "category": row['전문일반구분']
    }
    for _, row in df.iterrows()
]
ids = [str(i) for i in range(len(df))]

# 5000개 단위로 add하기
for i in range(0, len(documents), batch_size):
    collection.add(
        documents=documents[i:i+batch_size],
        metadatas=metadatas[i:i+batch_size],
        ids=ids[i:i+batch_size]
    )
    print(f"Added {i+len(documents[i:i+batch_size])} / {len(documents)}")

Added 5000 / 27387
Added 10000 / 27387
Added 15000 / 27387
Added 20000 / 27387
Added 25000 / 27387
Added 27387 / 27387


In [7]:
result = collection.query(
    query_texts=["페라트"],
    n_results=5
)

result

{'ids': [['0', '26967', '2384', '26384', '4779']],
 'distances': [[0.33144474, 0.33241343, 0.34661198, 0.34680337, 0.35454622]],
 'embeddings': None,
 'metadatas': [[{'category': '전문의약품', 'title': 4210},
   {'title': '02180', 'category': '전문의약품'},
   {'title': 2560, 'category': '일반의약품'},
   {'category': '전문의약품', 'title': '04210'},
   {'category': '전문의약품', 'title': 1140}]],
 'documents': [['페라트라정2.5밀리그램(레트로졸)',
   '페타바정(피타바스타틴|페노피브레이트)',
   '페로나정',
   '페트라정(레트로졸)',
   '페인리스정']],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [11]:
client_db.list_collections()

[Collection(name=test1),
 Collection(name=book_collection),
 Collection(name=book_collection1)]

In [12]:
import requests

MODEL_URL = "http://192.168.0.12:11435/api/generate"

payload = {
    "model" : "llama-3.1-ko:latest",
    "prompt" : "부경대학교에 대해서 설명해줘",
    "stream" : False
}

response = requests.post(MODEL_URL, json=payload)

if response.status_code == 200:
    result = response.json()
    print(f"답변:{result['response']}")

답변:부경대학교는 대한민국의 사립 대학으로, 1992년에 설립되었습니다. 현재 부천시와 광명시에 캠퍼스가 있으며, 총 19개 학과와 3개 대학원을 운영하고 있습니다. 또한, 다양한 연구센터와 협동연구기관도 있습니다.


In [14]:
from requests.api import request
import requests

MODEL_URL = "http://192.168.0.12:11435/api/generate"

# RAG
context = "이 부분은 DB에서 조회한 최신정보로 구성"
question = "해열제에 대해서 설명해줘"

prompt = f"""
    너는 약에 대한 전문가다
    아래 문서를 참고해서 답변해

    문서:
    { context }

    질문:
    { question }

    답변:
"""

payload = {
    "model" : "exaone-3.5:latest", #"llama-3.1-ko:latest",
    "prompt" : prompt,
    "stream" : False
}

response = requests.post(MODEL_URL, json=payload)

if response.status_code == 200:
    result = response.json()
    print(f"답변:{result['response']}")

답변:물론입니다. 해열제에 대해 자세히 설명해 드리겠습니다.

### 해열제란?
해열제는 체온이 정상 범위를 초과할 때 체온을 낮추는 데 사용되는 약물입니다. 주로 발열로 인한 불편함이나 건강 위험을 줄이기 위해 사용됩니다. 발열은 감염이나 다양한 질병의 자연스러운 반응일 수 있지만, 때로는 체온이 너무 높아지면 불편함이나 합병증을 초래할 수 있습니다. 따라서 적절한 해열제 사용은 증상 완화에 도움이 됩니다.

### 주요 성분 및 종류
해열제의 주요 성분은 다음과 같습니다:

1. **아세트아미노펜 (Paracetamol)**
   - **효과**: 주로 통증 완화와 중등도 발열 감소에 효과적입니다.
   - **사용 예**: 타이레놀 (Tylenol) 등
   - **특징**: 간 기능에 주의가 필요하며, 과다 복용 시 간 손상 위험이 있습니다.

2. **이부프로펜 (Ibuprofen)**
   - **효과**: 통증 완화와 함께 중등도에서 중증 발열 감소에 효과적입니다.
   - **사용 예**: 어드비아 (Advil), 모티프렌 (Motrin) 등
   - **특징**: 위장관 부작용 가능성이 있으며, 심혈관 질환이 있는 경우 주의가 필요합니다.

3. **니페록사이드 (Nifedipine)** (주로 혈압 조절제로 알려져 있지만 일부 상황에서 사용)
   - **주의**: 일반적인 해열제로는 덜 흔하며, 특정 상황에서 사용될 수 있습니다. 일반적인 해열제 목록에서는 주로 제외됩니다.

### 복용법 안내
- **용량**: 각 약물의 정확한 용량은 연령, 체중, 그리고 특정 건강 상태에 따라 달라집니다. 반드시 제품 라벨이나 의사의 지시를 따르세요.
  - **아세트아미노펜**: 일반적으로 성인의 경우 325-650mg을 4-6시간마다 복용합니다.
  - **이부프로펜**: 성인의 경우 200-400mg을 4-6시간마다 복용합니다.
  
- **복용 시기**: 식사 후 복용하면 위장 장애를 줄일 수 있습니다. 하지만 특정 약물은 공복에 복용해야 할 수